<a href="https://colab.research.google.com/github/sadot04/Inteligencia_artificial/blob/main/Insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from tensorflow.keras import layers, models, callbacks

from sklearn.preprocessing import LabelEncoder

In [ ]:
CSV_PATH = "insurance.csv"
df = pd.read_csv(CSV_PATH)

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Tensorflow: ", tf.__version__)

Tensorflow:  2.19.0


In [ ]:
print("Forma del dataset:", df.shape)
print("Columnas:", list(df.columns))

Forma del dataset: (1338, 7)
Columnas: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']


In [ ]:
le = LabelEncoder()
df["sex"] = le.fit_transform(df["sex"])
df["smoker"] = le.fit_transform(df["smoker"])
df["region"] = le.fit_transform(df["region"])

In [ ]:
y = df["smoker"].astype(float).values

In [ ]:
X = df[[
  "age","sex","bmi","children","region","charges"
]].copy()
#Se usan las siguientes columnas para predecir el porcentaje de que una persona fume o no

In [ ]:
num_cols = ["age","bmi","children","charges"]
cat_cols = ["region", "sex"]



In [ ]:
# 5) Preprocesamiento con ColumnTransformer
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)


In [ ]:
# 6) Split train/test estratificado
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

In [ ]:
# Ajustar transformadores en train y transformar ambos
X_train = preprocess.fit_transform(X_train_df)
X_test = preprocess.transform(X_test_df)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Dimensiones de entrada:", X_train.shape[1])

# 7) Definir y compilar el modelo (MLP para clasificación binaria)
def build_model(input_dim: int) -> tf.keras.Model:
    model = models.Sequential([
#En este caso utilizamos 3 capas, 2 de 32 neuronas y una de 1, con siendo un total de 65 neuronas
#Se tomaron en cuenta estas cantidades porque nos dieron buenos resultados en el coeficiente de determinacion
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.15),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.15),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model
model = build_model(X_train.shape[1])
model.summary()

# 8) Callbacks para entrenamiento
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=12,
                            restore_best_weights=True),
    callbacks.ModelCheckpoint("weather_best.keras", monitor="val_loss", mode="min",
                              save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6)
]

# 9) Entrenamiento
hist = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=cbs,
    verbose=1
)

# 10) Evaluación en test
y_pred_proba = model.predict(X_test).ravel()

print("\nError Cuadrático Medio (MSE):", mean_squared_error(y_test, y_pred_proba))
print("Error Absoluto Medio (MAE):", mean_absolute_error(y_test, y_pred_proba))
print("Coeficiente de Determinación (R²):", r2_score(y_test, y_pred_proba))

def predict_one(sample: dict) -> float:

  """Recibe un diccionario 'crudo' con las llaves esperadas:
  age, sex, bmi, children, smoker, region, charges
  """
  #Convertir a DataFrame con columnas en orden esperado
  s = pd.DataFrame([sample])

  #Aplicar Exactamente el mismo procesamiento
  s_proc = preprocess.transform(s[X.columns])
  s_proc = s_proc.astype("float32")

  #Predecir
  proba = model.predict(s_proc).item()
  return proba

sample = {
    "age": 19,
    "sex": 0,
    "bmi": 27.9,
    "children": 0,
    "region": 2,
    "charges": 16800
}

proba = predict_one(sample)
print(f"Probabilidad de fumador: {proba:.4f}")

Dimensiones de entrada: 10


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,441 (5.63 KB)

 Trainable params: 1,441 (5.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4966 - loss: 0.7275 - val_accuracy: 0.8318 - val_loss: 0.5273 - learning_rate: 0.0010
Epoch 2/200
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7998 - loss: 0.5199 - val_accuracy: 0.8364 - val_loss: 0.3974 - learning_rate: 0.0010
Epoch 3/200
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8295 - loss: 0.4125 - val_accuracy: 0.9065 - val_loss: 0.2932 - learning_rate: 0.0010
Epoch 4/200
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8687 - loss: 0.3164 - val_accuracy: 0.9346 - val_loss: 0.2137 - learning_rate: 0.0010
Epoch 5/200
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8842 - loss: 0.2609 - val_accuracy: 0.9299 - val_loss: 0.1642 - learning_rate: 0.0010
Epoch 6/200
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9031 - loss: 0.2165 - val_accuracy: 0.9439 - val_loss: 0.1359 - learning_rate: 0.0010
Epoch 7/200
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9063 - loss: 0.1811 - val_ac